# MechVerify V4 — Qwen extraction + Python verification
This is a **development-set experiment**, not a held-out scientific result. No LLM API key or locally installed model is required. Select **Runtime → Change runtime type → T4 GPU** if available.
The model parses **raw claims only** using an evidence-derived ontology. It does not see gold labels or human preannotated claim structures. Python then compares the extracted features to curated evidence.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
results_dir = Path('/content/drive/MyDrive/MechVerify-COF/results')
results_dir.mkdir(parents=True, exist_ok=True)
print('Results will be saved in:', results_dir)


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none: Qwen may be slow')


In [ ]:
import subprocess
repo_dir = Path('/content/MechVerify-COF')
if not repo_dir.exists():
    subprocess.run(['git','clone','https://github.com/bao-yutian/MechVerify-COF.git',str(repo_dir)],check=True)
else:
    subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only'],check=True)
%cd /content/MechVerify-COF
!python -m unittest discover -s tests -v
%pip install -q -U transformers accelerate


## Run three cases first
Model weights are downloaded into the Colab runtime on first use. The JSON file is updated after each finished claim. If execution stops, partial results remain in Drive; the script currently does not resume automatically.


In [ ]:
!python -u -m mechverify.v4_qwen --limit 3 --output /content/drive/MyDrive/MechVerify-COF/results/v4_smoke.json


## Full 15-case development run
Run after the smoke test works. It reloads Qwen from local Colab cache, so there may be another model-loading step.


In [ ]:
!python -u -m mechverify.v4_qwen --output /content/drive/MyDrive/MechVerify-COF/results/v4_all_15.json


## Examine failures and extraction quality
Read saved JSON without loading Qwen. Pay particular attention to property and route mismatches, and to extraction errors versus insufficient evidence.


In [ ]:
import json
result_path = results_dir / 'v4_all_15.json'
if result_path.exists():
    report = json.loads(result_path.read_text(encoding='utf-8'))
    print(json.dumps(report.get('summary', {}), indent=2, ensure_ascii=False))
    for item in report['predictions']:
        if item['predicted'] != item['gold_label']:
            print('\n',item['id'], item['predicted'], 'vs', item['gold_label'])
            print('Extraction:',item['extracted'])
            print('Reason:',item['reason'])
else:
    print('No full V4 report yet.')


## Replay saved extraction without GPU
If the Python verifier changes later, re-evaluate saved extractions without downloading the Qwen model again. Replay is only valid for **V4 extraction JSON**, not older Qwen verdict-only JSON.


In [ ]:
# Optional CPU-only replay; uncomment when a v4_all_15.json result exists.
# !python -m mechverify.v4_qwen --replay /content/drive/MyDrive/MechVerify-COF/results/v4_all_15.json --output /content/drive/MyDrive/MechVerify-COF/results/v4_replay.json
